<a href="https://colab.research.google.com/github/CristianCarrereAlvarez/monitor-mercado-laboral/blob/main/SMLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitor Mercado Laboral Chile

**Este notebook es para preguntas nuevas.** Todo lo que se corre igual
todos los meses vive en scripts, porque un cuaderno es la herramienta
equivocada para algo que se ejecuta siempre idéntico.

## Todo el ciclo mensual va en la terminal

```bash
cd ~/monitor-mercado-laboral && git pull

./mensual.sh          # 10 áreas + consolidación + control de calidad
./capturar.sh "Salud"  # una sola área

python3 control.py    --maestras <datos>/maestras   # los chequeos, sueltos
python3 homologar.py  --maestras <datos>/maestras   # la cola de homologación
```

`mensual.sh` ya consolida y corre el control al terminar, y deja todo en
un log en `<datos>/logs/`. La captura **no corre en Colab**: Akamai
bloquea los rangos de datacenter y devuelve 403 en todo.

---

## Lo que sigue acá

Cargar las maestras y hacerles preguntas que todavía no tienen respuesta.
Si una pregunta se vuelve rutina, migrala a `control.py`.


---

## 1. Preparación

Las dos celdas, en orden. Colab borra su disco al cerrar, así que hay
que repetirlas al reabrir. Si corrés Jupyter local, saltealas: los datos
ya están en tu máquina.


### 1.1 Drive — define `DATOS`

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATOS = '/content/drive/MyDrive/monitor_mercado_laboral'
os.makedirs(f'{DATOS}/crudo', exist_ok=True)
os.makedirs(f'{DATOS}/maestras', exist_ok=True)

print('crudo   :', sorted(os.listdir(f'{DATOS}/crudo')))
print('maestras:', sorted(os.listdir(f'{DATOS}/maestras')))

### 1.2 Repositorio

**Cuándo hace falta:** solo si vas a usar el catálogo SIES, o sea
`import carreras_sies_2026`.

Las maestras son CSV y pandas las lee sin ayuda de nadie. Para explorar
`avisos`, `empresas` o `aviso_carrera` esta celda **no aporta nada** y la
podés saltear.

El catálogo sirve cuando la pregunta necesita el **universo** y no lo
**observado**: qué carreras de un área nunca aparecieron en ningún aviso,
a qué área SIES pertenece un término, cuántos términos tiene cada área.
Eso no está en las maestras, que solo registran lo que el sitio devolvió.

El clon es efímero y de solo lectura: Colab borra `/content` al cerrar,
por eso se vuelve a clonar en cada sesión. Los datos nunca viven acá.


In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/CristianCarrereAlvarez/monitor-mercado-laboral.git'

if not os.path.exists('/content/repo'):
    !git clone $REPO /content/repo
else:
    !git -C /content/repo pull --ff-only

# Sin esta línea el clon está en disco pero `import` no lo encuentra:
# el cwd de Colab es /content, no /content/repo.
if '/content/repo' not in sys.path:
    sys.path.insert(0, '/content/repo')

print(subprocess.run(['git', '-C', '/content/repo', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

# El import es la verificación: si esto corre, la celda cumplió su función.
from carreras_sies_2026 import CARRERAS_POR_AREA
print(f'catálogo: {len(CARRERAS_POR_AREA)} áreas, '
      f'{sum(len(c) for c in CARRERAS_POR_AREA.values())} carreras')


**Si `git pull` falla** con *"untracked working tree file would be overwritten"*,
re-cloná limpio con esta celda (es seguro: en el repo no hay datos):

In [ ]:
import shutil
shutil.rmtree('/content/repo', ignore_errors=True)
!git clone $REPO /content/repo
!ls /content/repo

---

## 2. Cargar las maestras

El punto de partida de cualquier exploración.


In [ ]:
import pandas as pd

M = f'{DATOS}/maestras'
avisos    = pd.read_csv(f'{M}/avisos.csv')
carreras  = pd.read_csv(f'{M}/aviso_carrera.csv')
terminos  = pd.read_csv(f'{M}/aviso_termino.csv')
empresas  = pd.read_csv(f'{M}/empresas.csv')
taxonomia = pd.read_csv(f'{M}/carreras_trabajando.csv')

for n, d in [('avisos', avisos), ('aviso_carrera', carreras),
             ('aviso_termino', terminos), ('empresas', empresas),
             ('carreras_trabajando', taxonomia)]:
    print(f'{n:22s} {d.shape[0]:>7,} filas × {d.shape[1]} columnas')

### Tres reglas que condicionan cualquier análisis

No son sugerencias: sin ellas los números salen mal.

1. **Filtrá `fuente == 'declarada'`** en `aviso_carrera`. Las filas
   `keyword_only` son trazabilidad, no evidencia: el término de búsqueda
   no es atribución de carrera.
2. **Excluí los avisos genéricos** (`n_carreras_declaradas > 30`). Diez
   avisos de un mismo empleador declaran 504 carreras y aparecen en
   cualquier búsqueda.
3. **Deduplicá por empleador antes de leer un agregado.** En Derecho un
   solo empleador fue el 35% del área. El conteo de avisos mide
   publicación, no demanda.

`control.py` reporta las tres cosas después de cada consolidación.


In [ ]:
# Base limpia para trabajar: sin genéricos, solo carreras declaradas
esp = avisos[avisos.n_carreras_declaradas <= 30]
dec = carreras[carreras.fuente == 'declarada']
dec = dec[dec.aviso_id.isin(esp.aviso_id)]

print(f'avisos totales    : {len(avisos):,}')
print(f'avisos específicos: {len(esp):,}')
print(f'pares aviso×carrera declarados: {len(dec):,}')

---

## 3. Tu pregunta acá

Espacio libre. Si algo de lo que escribas termina siendo un chequeo que
querés repetir todos los meses, su lugar es `control.py`.
